In [1]:
import argparse
import time
import torch
import torch.nn as nn

# Runtime metrics used by TransformerBlock.  Keep these global so the
# notebook can inspect FFN cost without requiring a profiler.
FFN_TIME_MS = []
FFN_MEM_BYTES = []

In [2]:
#####################################
# NEW: GQA instead of MHA
#####################################
class GroupedQueryAttention(nn.Module):
    def __init__(
            self, d_in, d_out, dropout, num_heads, num_kv_groups, dtype=None, qkv_bias=False
    ):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_key = nn.Linear(d_in, num_kv_groups * self.head_dim, bias=qkv_bias, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * self.head_dim, bias=qkv_bias, dtype=dtype)
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias, dtype=dtype)
        self.out_proj = nn.Linear(d_out, d_out, bias=False, dtype=dtype)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)
        self.ptr_current_pos = 0

    def forward(self, x, use_cache=False):
        b, num_tokens, _ = x.shape

        # Apply projections
        queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)

        # Reshape
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys_new = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values_new = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        if use_cache:
            if self.cache_k is None:
                self.cache_k, self.cache_v = keys_new, values_new
            else:
                self.cache_k = torch.cat([self.cache_k, keys_new], dim=2)
                self.cache_v = torch.cat([self.cache_v, values_new], dim=2)
            keys_base, values_base = self.cache_k, self.cache_v
        else:
            keys_base, values_base = keys_new, values_new
            if self.cache_k is not None or self.cache_v is not None:
                self.cache_k, self.cache_v = None, None
                self.ptr_current_pos = 0

        # Expand keys and values to match the number of heads
        # Shape: (b, num_heads, num_tokens, head_dim)
        keys = keys_base.repeat_interleave(self.group_size, dim=1)  # Shape: (b, num_heads, num_tokens, head_dim)
        values = values_base.repeat_interleave(self.group_size, dim=1)  # Shape: (b, num_heads, num_tokens, head_dim)
        # For example, before repeat_interleave along dim=1 (query groups):
        #   [K1, K2]
        # After repeat_interleave (each query group is repeated group_size times):
        #   [K1, K1, K2, K2]
        # If we used regular repeat instead of repeat_interleave, we'd get:
        #   [K1, K2, K1, K2]

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        # Shape: (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        ####################################################
        # causal mask
        num_tokens_Q = queries.shape[-2]
        num_tokens_K = keys.shape[-2]
        device = queries.device
        if use_cache:
            q_positions = torch.arange(
                self.ptr_current_pos,
                self.ptr_current_pos + num_tokens_Q,
                device=device,
                dtype=torch.long,
            )
            self.ptr_current_pos += num_tokens_Q
        else:
            q_positions = torch.arange(num_tokens_Q, device=device, dtype=torch.long)
            self.ptr_current_pos = 0
        k_positions = torch.arange(num_tokens_K, device=device, dtype=torch.long)
        mask = q_positions.unsqueeze(-1) < k_positions.unsqueeze(0)

        # Use the mask to fill attention scores
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        assert keys.shape[-1] == self.head_dim
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection

        return context_vec

    def reset_cache(self):
        self.cache_k, self.cache_v = None, None
        self.ptr_current_pos = 0

In [5]:
import torch

torch.manual_seed(42)

# ============================================================
# Giả sử câu:
# "I love AI"
#
# token 0 = "I"
# token 1 = "love"
# token 2 = "AI"
#
# Mỗi token được biểu diễn bằng vector 8 chiều.
# batch_size = 1 vì chỉ có 1 câu.
# seq_len = 3 vì có 3 token.
# d_in = 8 vì mỗi token có 8 feature.
# ============================================================

x = torch.randn(1, 3, 8)

mha = GroupedQueryAttention(
    d_in=8,
    d_out=8,
    num_heads=8,
    num_kv_groups=2,
    dropout=0.0,
)

mha.eval()

# Vì:
# d_out = 8
# num_heads = 2
#
# => head_dim = 8 / 2 = 4
#
# Mỗi token sau Wk/Wv sẽ từ:
# (8 feature)
#
# tách thành:
# head 0: 4 feature
# head 1: 4 feature


# ============================================================
# Decode từng token giống lúc LLM generate
# ============================================================

mha.reset_cache()

tokens = ["I", "love", "AI"]

for i in range(3):

    # Lấy đúng 1 token:
    #
    # x_i.shape = (1, 1, 8)
    #
    # 1 đầu tiên = batch size
    # 1 thứ hai   = đang xử lý 1 token
    # 8           = vector embedding/token có 8 chiều
    x_i = x[:, i:i+1, :]
    out = mha(x_i, use_cache=True)

    print(f"\nToken {i}: '{tokens[i]}'")

    # cache_k có shape:
    #
    # (batch, số_token_đã_cache, num_heads, head_dim)
    #
    # Ví dụ token đầu:
    # (1, 1, 2, 4)
    #
    # nghĩa là:
    # 1 câu
    # 1 token đã lưu
    # 2 attention heads
    # mỗi head có K vector 4 chiều
    print("K cache:", mha.cache_k.shape)
    print("V cache:", mha.cache_v.shape)


Token 0: 'I'
K cache: torch.Size([1, 2, 1, 1])
V cache: torch.Size([1, 2, 1, 1])

Token 1: 'love'
K cache: torch.Size([1, 2, 2, 1])
V cache: torch.Size([1, 2, 2, 1])

Token 2: 'AI'
K cache: torch.Size([1, 2, 3, 1])
V cache: torch.Size([1, 2, 3, 1])


In [6]:
import torch
import torch.nn as nn
#####################################
# Chapter 4
#####################################
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.emb_dim = emb_dim
        self.weight = nn.Parameter(torch.ones(emb_dim)).float()

    def forward(self, x):
        means = x.pow(2).mean(dim=-1, keepdim=True)
        x_normed = x * torch.rsqrt(means + self.eps)
        return (x_normed * self.weight).to(dtype=x.dtype)


In [7]:

torch.manual_seed(123)

example_batch = torch.randn(2, 3, 4)

rms_norm = RMSNorm(emb_dim=example_batch.shape[-1])
rmsnorm_pytorch = torch.nn.RMSNorm(example_batch.shape[-1], eps=1e-5)

assert torch.allclose(rms_norm(example_batch), rmsnorm_pytorch(example_batch))

In [8]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

# class FeedForward(nn.Module):
#     def __init__(self, cfg):
#         super().__init__()
#         self.layers = nn.Sequential(
#             nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
#             GELU(),
#             nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
#         )

#     def forward(self, x):
#         return self.layers(x)


In [9]:
class SiLU(nn.Module):
    def __init__(self):
        super(SiLU, self).__init__()

    def forward(self, x):
        return x * torch.sigmoid(x)
silu = SiLU()

assert torch.allclose(silu(example_batch), torch.nn.functional.silu(example_batch))


# Uses SwiGLU instead of GeLU to make it more comparable to MoE
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate_proj = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.value_proj = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.out_proj = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)
        self.silu = SiLU()

    def forward(self, x):
        x_gate = self.gate_proj(x)
        x_value = self.value_proj(x)
        x = self.silu(x_gate) * x_value
        return self.out_proj(x)

In [10]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            num_kv_groups=cfg["n_kv_groups"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x, use_cache=False):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)

        # x = self.att(x)   # Shape [batch_size, num_tokens, emb_size]
        ####################################################
        #  KV cache-related
        x = self.att(x, use_cache=use_cache)
        ####################################################

        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        use_cuda = torch.cuda.is_available()
        if use_cuda:
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            base_mem = torch.cuda.memory_allocated()
        start = time.perf_counter()
        x = self.ff(x)
        if use_cuda:
            torch.cuda.synchronize()
            peak_mem = torch.cuda.max_memory_allocated()
            FFN_MEM_BYTES.append(peak_mem - base_mem)
        FFN_TIME_MS.append((time.perf_counter() - start) * 1000.0)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x

In [21]:
class GPTModel(nn.Module):
    '''Small GPT language model using the MHA/KV-cache blocks above.'''
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'], dtype=cfg['dtype'])
        self.position_embedding = nn.Embedding(cfg['context_length'], cfg['emb_dim'], dtype=cfg['dtype'])
        self.dropout = nn.Dropout(cfg['drop_rate'])
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg['n_layers'])])
        self.final_norm = LayerNorm(cfg['emb_dim'])
        self.lm_head = nn.Linear(cfg['emb_dim'], cfg['vocab_size'], bias=False, dtype=cfg['dtype'])
        self.lm_head.weight = self.token_embedding.weight

    def reset_cache(self):
        for block in self.blocks:
            block.att.reset_cache()

    def forward(self, input_ids, use_cache=False):
        if input_ids.ndim != 2 or input_ids.shape[1] == 0:
            raise ValueError('input_ids must have shape (batch, tokens) and be non-empty')
        batch, tokens = input_ids.shape
        if tokens > self.cfg['context_length']:
            raise ValueError('input is longer than context_length')
        if use_cache and self.blocks[0].att.cache_k is not None:
            past_tokens = self.blocks[0].att.cache_k.shape[-2]
        else:
            past_tokens = 0
        if past_tokens + tokens > self.cfg['context_length']:
            raise ValueError('cached input exceeds context_length; reset_cache() first')
        positions = torch.arange(past_tokens, past_tokens + tokens, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions).unsqueeze(0)
        x = self.dropout(x)
        for block in self.blocks:
            x = block(x, use_cache=use_cache)
        return self.lm_head(self.final_norm(x))

    @torch.inference_mode()
    def generate(self, input_ids, max_new_tokens, use_cache=True):
        output = input_ids.clone()
        self.eval()
        self.reset_cache()
        if use_cache:
            logits = self(output[:, -self.cfg['context_length']:], use_cache=True)
            for _ in range(max_new_tokens):
                next_token = logits[:, -1].argmax(dim=-1, keepdim=True)
                output = torch.cat((output, next_token), dim=1)
                if output.shape[1] >= self.cfg['context_length']:
                    self.reset_cache()
                    logits = self(output[:, -self.cfg['context_length']:], use_cache=True)
                else:
                    logits = self(next_token, use_cache=True)
        else:
            for _ in range(max_new_tokens):
                logits = self(output[:, -self.cfg['context_length']:])
                output = torch.cat((output, logits[:, -1].argmax(dim=-1, keepdim=True)), dim=1)
        return output

cfg = {
    'vocab_size': 49152,
    'context_length': 8192,

    'emb_dim': 576,
    'hidden_dim': 1536,   # ← không phải 4*576 = 2304

    'n_heads': 9,
    'n_kv_groups': 3,     # 3 Q heads share 1 KV head
                          # => n_kv_heads = 9 // 3 = 3

    'n_layers': 30,       # ← SmolLM2-135M thật là 30 layers

    'drop_rate': 0.0,
    'qkv_bias': False,

    'dtype': torch.float32,  # để học/debug thì FP32 OK
}
torch.manual_seed(123)
model = GPTModel(cfg).eval()
prompt = torch.randint(0, cfg['vocab_size'], (1, 8))
full_logits = model(prompt)
model.reset_cache()
_ = model(prompt[:, :4], use_cache=True)
cached_logits = model(prompt[:, 4:], use_cache=True)
assert torch.allclose(full_logits[:, 4:], cached_logits, atol=1e-4, rtol=1e-4)
model.reset_cache()
uncached = model.generate(prompt, max_new_tokens=4, use_cache=False)
cached = model.generate(prompt, max_new_tokens=4, use_cache=True)
assert torch.equal(uncached, cached)
print('model ready; logits:', tuple(full_logits.shape), 'generation parity: OK')

model ready; logits: (1, 8, 49152) generation parity: OK


In [22]:
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Model size:       {total_params / 1e6:.3f}M")

Total params:     139,268,736
Trainable params: 139,268,736
Model size:       139.269M
